In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.6 MB/s eta 0:00:00


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn.functional as F
import pydicom
from transformers import AutoTokenizer, AutoModel
from skimage.transform import resize

In [ ]:
# ==============================
# PATHS (UPDATE IF NEEDED)
# ==============================
BASE_PATH = "/content/drive/My Drive/MediqaRadar02"
IMAGE_PATH = os.path.join(BASE_PATH, "image_dev")
REPORT_PATH = os.path.join(BASE_PATH, "dev_preliminary_report")
EDITS_JSON = os.path.join(BASE_PATH, "dev_edits.json")

In [ ]:
# ==============================
# LOAD EDITS
# ==============================
with open(EDITS_JSON, "r") as f:
    edits_data = json.load(f)

print(f"Loaded {len(edits_data)} cases")


Loaded 20 cases


In [ ]:
GT_PATH = "/content/drive/My Drive/RADAR-main/RADAR-main/eval/groundtruth_dev.csv"

In [ ]:
import pandas as pd
with open(EDITS_JSON, "r") as f:
    edits_data = json.load(f)

gt_df = pd.read_csv(GT_PATH)

agreement_map = {"agree":0, "partially agree":1, "disagree":2}
severity_map  = {"critical":0, "moderate":1, "negligible":2}
edit_map      = {"correction":0, "addition":1, "clarification":2}

gt_df["agreement_label"] = gt_df["agreement"].map(agreement_map)
gt_df["severity_label"]  = gt_df["severity"].map(severity_map)
gt_df["edit_label"]      = gt_df["edit_type"].map(edit_map)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
text_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DebertaV2Model(
  (embeddings): DebertaV2Embeddings(
    (word_embeddings): Embedding(128100, 768, padding_idx=0)
    (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): DebertaV2Encoder(
    (layer): ModuleList(
      (0-11): 12 x DebertaV2Layer(
        (attention): DebertaV2Attention(
          (self): DisentangledSelfAttention(
            (query_proj): Linear(in_features=768, out_features=768, bias=True)
            (key_proj): Linear(in_features=768, out_features=768, bias=True)
            (value_proj): Linear(in_features=768, out_features=768, bias=True)
            (pos_dropout): Dropout(p=0.1, inplace=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): DebertaV2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

In [ ]:
def get_text_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = text_model(**inputs)

    emb = outputs.last_hidden_state[:, 0, :]
    emb = F.normalize(emb, p=2, dim=1)

    return emb.cpu().numpy().squeeze().astype(np.float32)

In [ ]:
def load_ct_series(case_id, series_name="ABD_PEL", max_slices=50, img_size=128):

    path = os.path.join(IMAGE_PATH, str(case_id), series_name)
    if not os.path.exists(path):
        return None

    files = [os.path.join(path, f) for f in os.listdir(path)]

    slices = []
    for f in files:
        try:
            ds = pydicom.dcmread(f)
            slices.append(ds)
        except:
            continue

    if len(slices) == 0:
        return None

    slices.sort(key=lambda x: getattr(x, "InstanceNumber", 0))

    imgs = np.array([
        s.pixel_array.astype(np.float32) *
        getattr(s, "RescaleSlope", 1.0) +
        getattr(s, "RescaleIntercept", 0.0)
        for s in slices
    ])

    imgs = np.clip(imgs, -1000, 1000)
    imgs = (imgs - imgs.min()) / (imgs.max() - imgs.min() + 1e-6)

    imgs = np.stack([resize(sl, (img_size, img_size)) for sl in imgs])

    if imgs.shape[0] >= max_slices:
        idx = np.linspace(0, imgs.shape[0]-1, max_slices).astype(int)
        imgs = imgs[idx]
    else:
        pad = max_slices - imgs.shape[0]
        imgs = np.concatenate([imgs, np.zeros((pad, img_size, img_size))], axis=0)

    return imgs

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom
from transformers import AutoTokenizer, AutoModel
from skimage.transform import resize

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(1, 8, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(8, 16, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)
        )
        self.fc = nn.Linear(16, 512)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

cnn_model = SimpleCNN().to(device)
cnn_model.eval()

SimpleCNN(
  (conv): Sequential(
    (0): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): ReLU()
    (2): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): ReLU()
    (5): AdaptiveAvgPool3d(output_size=1)
  )
  (fc): Linear(in_features=16, out_features=512, bias=True)
)

In [ ]:
def get_text_embedding(text):
    # ==============================
    # FIX: HANDLE EMPTY TEXT
    # ==============================
    if text is None or len(text.strip()) == 0:
        return np.zeros(768, dtype=np.float32)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    # SAFETY CHECK
    if inputs["input_ids"].shape[1] == 0:
        return np.zeros(768, dtype=np.float32)

    with torch.no_grad():
        outputs = text_model(**inputs)

    emb = outputs.last_hidden_state[:, 0, :]
    emb = F.normalize(emb, p=2, dim=1)

    return emb.cpu().numpy().squeeze().astype(np.float32)

In [ ]:
train_data = []

for case in edits_data:
    case_id = case["case_id"]

    try:
        report_text = open(f"{REPORT_PATH}/{case_id}.txt").read()
    except:
        report_text = ""

    report_emb = get_text_embedding(report_text)

    ct_volume = load_ct_series(case_id)
    if ct_volume is not None:
        ct_tensor = torch.tensor(ct_volume).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            ct_emb = cnn_model(ct_tensor).cpu().numpy().squeeze()
    else:
        ct_emb = np.zeros(512)

    ct_emb = ct_emb / (np.linalg.norm(ct_emb) + 1e-6)

    for edit in case["edits"]:
        edit_id = edit["edit_id"]

        row = gt_df[gt_df["edit_id"] == edit_id]
        if len(row) == 0:
            continue
        row = row.iloc[0]

        edit_emb = get_text_embedding(edit["suggested_edit_text"])

        text = np.concatenate([
            report_emb / (np.linalg.norm(report_emb)+1e-6),
            edit_emb / (np.linalg.norm(edit_emb)+1e-6)
        ])

        x = np.concatenate([text, ct_emb])

        train_data.append({
            "x": x.astype(np.float32),
            "y": (
                int(row["agreement_label"]),
                int(row["severity_label"]),
                int(row["edit_label"])
            )
        })

In [ ]:
class RadarDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.X = torch.tensor([d["x"] for d in data])
        self.y = torch.tensor([d["y"] for d in data])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

loader = torch.utils.data.DataLoader(RadarDataset(train_data), batch_size=8, shuffle=True)

/tmp/ipykernel_2226/4075959649.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.X = torch.tensor([d["x"] for d in data])


In [ ]:
class FusionModel(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU()
        )

        self.a = nn.Linear(512, 3)
        self.s = nn.Linear(512, 3)
        self.e = nn.Linear(512, 3)

    def forward(self, x):
        x = self.shared(x)
        return self.a(x), self.s(x), self.e(x)

model = FusionModel(train_data[0]["x"].shape[0]).to(device)

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    total = 0
    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        opt.zero_grad()
        a, s, e = model(X)

        loss = (
            1.5*loss_fn(a, y[:,0]) +
            loss_fn(s, y[:,1]) +
            loss_fn(e, y[:,2])
        )

        loss.backward()
        opt.step()
        total += loss.item()

    print("Epoch", epoch+1, total)

Epoch 1 87.02340769767761
Epoch 2 82.21699666976929
Epoch 3 77.57434153556824
Epoch 4 76.29434156417847
Epoch 5 75.59356713294983


In [ ]:
model.eval()
results = []

for case in edits_data:
    case_id = case["case_id"]

    # ==============================
    # LOAD REPORT
    # ==============================
    try:
        report_text = open(f"{REPORT_PATH}/{case_id}.txt").read()
    except:
        report_text = ""

    report_emb = get_text_embedding(report_text)
    report_emb = report_emb / (np.linalg.norm(report_emb) + 1e-6)

    # ==============================
    # LOAD CT
    # ==============================
    ct_volume = load_ct_series(case_id)

    if ct_volume is not None:
        ct_tensor = torch.tensor(ct_volume, dtype=torch.float32)\
                        .unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            ct_emb = cnn_model(ct_tensor).cpu().numpy().squeeze().astype(np.float32)
    else:
        ct_emb = np.zeros(512, dtype=np.float32)

    ct_emb = ct_emb / (np.linalg.norm(ct_emb) + 1e-6)

    # ==============================
    # EDIT LOOP
    # ==============================
    for edit in case["edits"]:
        edit_text = edit.get("suggested_edit_text", "")

        edit_emb = get_text_embedding(edit_text)
        edit_emb = edit_emb / (np.linalg.norm(edit_emb) + 1e-6)

        # ==============================
        # FIX 1: consistent concat + dtype
        # ==============================
        x = np.concatenate([report_emb, edit_emb, ct_emb]).astype(np.float32)

        # ==============================
        # FIX 2: tensor dtype
        # ==============================
        X = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            a, s, e = model(X)

        results.append({
            "edit_id": edit["edit_id"],
            "agreement": a.argmax(dim=1).item(),
            "severity": s.argmax(dim=1).item(),
            "edit_type": e.argmax(dim=1).item()
        })

In [ ]:
df = pd.DataFrame(results)

In [ ]:
df = df.rename(columns={
    "agreement": "agreement_pred",
    "severity": "severity_pred",
    "edit_type": "edit_pred"
})

In [ ]:
eval_df = gt_df.merge(df, on="edit_id", how="inner")

In [ ]:
agreement_acc = (eval_df["agreement_label"] == eval_df["agreement_pred"]).mean()
severity_acc  = (eval_df["severity_label"]  == eval_df["severity_pred"]).mean()
edit_acc      = (eval_df["edit_label"]      == eval_df["edit_pred"]).mean()

In [ ]:
print("\n=== FINAL RESULTS ===")
print(f"Agreement : {agreement_acc:.4f}")
print(f"Severity  : {severity_acc:.4f}")
print(f"EditType  : {edit_acc:.4f}")
print(f"Average   : {(agreement_acc+severity_acc+edit_acc)/3:.4f}")


=== FINAL RESULTS ===
Agreement : 0.5978
Severity  : 0.5109
EditType  : 0.5870
Average   : 0.5652


In [ ]:
df = pd.DataFrame(results)
df.to_csv("/content/drive/My Drive/RADAR-main/RADAR-main/traincnnsubmission.csv", index=False)

# Merge with GT
eval_df = gt_df.merge(df, on="edit_id")

def acc(col):
    return (eval_df[col] == eval_df[col+"_pred"]).mean()

print("\n=== FINAL RESULTS ===")
print("Agreement:", acc("agreement"))
print("Severity :", acc("severity"))
print("EditType :", acc("edit_type"))


=== FINAL RESULTS ===
Agreement: 0.7445652173913043
Severity : 0.5543478260869565
EditType : 0.6032608695652174


In [ ]:
##ABOVE RESULT ACHIEVE ON VALID DATA RECHECK
df = pd.DataFrame(results)
df.to_csv("/content/drive/My Drive/RADAR-main/RADAR-main/traincnnsubmission.csv", index=False)

# Merge with GT
eval_df = gt_df.merge(df, on="edit_id")

def acc(col):
    return (eval_df[col] == eval_df[col+"_pred"]).mean()

print("\n=== FINAL RESULTS ===")
print("Agreement:", acc("agreement"))
print("Severity :", acc("severity"))
print("EditType :", acc("edit_type"))


=== FINAL RESULTS ===
Agreement: 0.7228260869565217
Severity : 0.5543478260869565
EditType : 0.6141304347826086


In [ ]:
# ==============================
# PATHS (UPDATE IF NEEDED)
# ==============================
TEST_BASE_PATH = "/content/drive/My Drive/MediqaRadar02Test"
TEST_IMAGE_PATH = os.path.join(TEST_BASE_PATH, "image_test")
TEST_REPORT_PATH = os.path.join(TEST_BASE_PATH, "test_preliminary_report")
TEST_EDITS_JSON = os.path.join(TEST_BASE_PATH, "test_edits.json")

In [ ]:
# ==============================
# LOAD EDITS
# ==============================
with open(TEST_EDITS_JSON, "r") as f:
    TEST_edits_data = json.load(f)

print(f"Loaded {len(TEST_edits_data)} cases")


Loaded 29 cases


In [ ]:
import json

# print dataset structure
print("Total cases:", len(TEST_edits_data))

# show first case
print("\n=== FIRST CASE ===")
print(TEST_edits_data[0])

# show edits inside it
print("\n=== EDITS OF FIRST CASE ===")
for i, edit in enumerate(TEST_edits_data[0]["edits"][:3]):
    print(f"\nEdit {i+1}:")
    print(edit)

Total cases: 29

=== FIRST CASE ===
{'case_id': 10001, 'clinical_indication': 'small bowel mass seen on push, eval for neoplasm, bleeding', 'preliminary_report_time': 'daytime', 'edits': [{'edit_id': '10001_01', 'suggested_edit_text': 'A few prominent mediastinal lymph nodes without adenopathy.'}, {'edit_id': '10001_02', 'suggested_edit_text': 'The prostate is markedly enlarged with a suspicious soft tissue mass replacing normal architecture.'}, {'edit_id': '10001_03', 'suggested_edit_text': 'There is a moderate right-sided pleural effusion.'}, {'edit_id': '10001_04', 'suggested_edit_text': 'Sclerotic foci in the right iliac bone and right sacral bone, likely representing bone islands, and mild degenerative changes of the spine are present.'}, {'edit_id': '10001_05', 'suggested_edit_text': 'Calcified granulomas in various locations in the lungs and a noncalcified nodule in the right upper lobe.'}, {'edit_id': '10001_06', 'suggested_edit_text': 'Minimal mesenteric stranding associated w

In [ ]:
sample = TEST_edits_data[0]

print("Keys in case:", sample.keys())
print("\nEdit keys:", sample["edits"][0].keys())

Keys in case: dict_keys(['case_id', 'clinical_indication', 'preliminary_report_time', 'edits'])

Edit keys: dict_keys(['edit_id', 'suggested_edit_text'])


In [ ]:
agreement_map = {
    0: "agree",
    1: "partially agree",
    2: "disagree"
}

severity_map = {
    0: "critical",
    1: "moderate",
    2: "negligible"
}

edit_map = {
    0: "correction",
    1: "addition",
    2: "clarification"
}

In [ ]:
model.eval()
results = []

for case in TEST_edits_data:
    case_id = case["case_id"]

    # ==============================
    # REPORT
    # ==============================
    try:
        report_text = open(f"{TEST_REPORT_PATH}/{case_id}.txt").read()
    except:
        report_text = ""

    report_emb = get_text_embedding(report_text)
    report_emb = report_emb / (np.linalg.norm(report_emb) + 1e-6)

    # ==============================
    # CT
    # ==============================
    ct_volume = load_ct_series(case_id)

    if ct_volume is not None:
        ct_tensor = torch.tensor(ct_volume, dtype=torch.float32)\
                        .unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            ct_emb = cnn_model(ct_tensor).cpu().numpy().squeeze().astype(np.float32)
    else:
        ct_emb = np.zeros(512, dtype=np.float32)

    ct_emb = ct_emb / (np.linalg.norm(ct_emb) + 1e-6)

    # ==============================
    # EDIT LOOP
    # ==============================
    for edit in case["edits"]:
        edit_text = edit.get("suggested_edit_text", "")

        edit_emb = get_text_embedding(edit_text)
        edit_emb = edit_emb / (np.linalg.norm(edit_emb) + 1e-6)

        # ==============================
        # FEATURE VECTOR
        # ==============================
        x = np.concatenate([report_emb, edit_emb, ct_emb]).astype(np.float32)
        X = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

        # ==============================
        # PREDICTION
        # ==============================
        with torch.no_grad():
            a, s, e = model(X)

        results.append({
            "edit_id": edit["edit_id"],
            "agreement": int(a.argmax(dim=1).item()),
            "severity": int(s.argmax(dim=1).item()),
            "edit_type": int(e.argmax(dim=1).item())
        })

In [ ]:
df = pd.DataFrame(results)
df.to_csv("/content/drive/My Drive/MediqaRadar02Test/testRobertasubmission.csv", index=False)

print("Saved submission.csv")
print(df.head())

Saved submission.csv
    edit_id  agreement  severity  edit_type
0  10001_01          2         2          1
1  10001_02          2         2          1
2  10001_03          0         2          1
3  10001_04          0         2          1
4  10001_05          2         2          1


In [ ]:
results

[{'edit_id': '10001_01',
  'agreement_pred': 0,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_02',
  'agreement_pred': 2,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_03',
  'agreement_pred': 0,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_04',
  'agreement_pred': 2,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_05',
  'agreement_pred': 0,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_06',
  'agreement_pred': 0,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_07',
  'agreement_pred': 2,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_08',
  'agreement_pred': 0,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_09',
  'agreement_pred': 2,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_10',
  'agreement_pred': 2,
  'severity_pred': 2,
  'edit_type_pred': 1},
 {'edit_id': '10001_11',
  'agreement_pred': 0,
  'severity_

In [ ]:
import pandas as pd

# load your saved file
df = pd.read_csv("/content/drive/MyDrive/MediqaRadar02Test/testRobertasubmission.csv")

# label maps (IMPORTANT: must match training)
agreement_map = {
    0: "agree",
    1: "partially agree",
    2: "disagree"
}

severity_map = {
    0: "critical",
    1: "moderate",
    2: "negligible"
}

edit_map = {
    0: "correction",
    1: "addition",
    2: "clarification"
}

# convert back to strings
df["agreement_pred"] = df["agreement"].map(agreement_map)
df["severity_pred"] = df["severity"].map(severity_map)
df["edit_type_pred"] = df["edit_type"].map(edit_map)

# save final submission file
df.to_csv("/content/drive/MyDrive/MediqaRadar02Test/testRobertasubmission_final.csv", index=False)

print(df.head())

    edit_id  agreement  severity  edit_type agreement_pred severity_pred  \
0  10001_01          2         2          1       disagree    negligible   
1  10001_02          2         2          1       disagree    negligible   
2  10001_03          0         2          1          agree    negligible   
3  10001_04          0         2          1          agree    negligible   
4  10001_05          2         2          1       disagree    negligible   

  edit_type_pred  
0       addition  
1       addition  
2       addition  
3       addition  
4       addition  


In [ ]:
df = pd.DataFrame(results)
df.to_csv("/content/drive/My Drive/MediqaRadar02Test/testRobertasubmission.csv", index=False)

print("Saved submission.csv")
print(df.head())

Saved submission.csv
    edit_id  agreement  severity  edit_type
0  10001_01          2         2          1
1  10001_02          2         2          1
2  10001_03          0         2          1
3  10001_04          0         2          1
4  10001_05          2         2          1


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/MediqaRadar02Test/TEST_BEST_RECHECKsubmission.csv", index=False)

In [ ]:
print(df["agreement_pred"].unique())
print(df["severity_pred"].unique())
print(df["edit_type_pred"].unique())

['agree' 'disagree']
['negligible' 'critical']
['addition' 'correction']


In [ ]:
import pandas as pd



# convert to JSON (list of records)
df.to_json("/content/drive/MyDrive/MediqaRadar02Test/submission.json", orient="records", indent=2)

print("Saved as submission.json")

Saved as submission.json
